In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "heart_disease.csv"
)

df_raw = pd.read_csv(raw_data_path)

df_clean = df_raw.copy()

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Working copy shape: {df_clean.shape}")
print(f"Raw file unchanged: {df_raw.equals(df_clean)}")

Raw dataset shape: (4238, 16)
Working copy shape: (4238, 16)
Raw file unchanged: True


In [16]:
column_name_mapping = {
    "Gender": "gender",
    "age": "age",
    "education": "education",
    "currentSmoker": "current_smoker",
    "cigsPerDay": "cigarettes_per_day",
    "BPMeds": "bp_meds",
    "prevalentStroke": "prevalent_stroke",
    "prevalentHyp": "prevalent_hypertension",
    "diabetes": "diabetes",
    "totChol": "total_cholesterol",
    "sysBP": "systolic_bp",
    "diaBP": "diastolic_bp",
    "BMI": "bmi",
    "heartRate": "heart_rate",
    "glucose": "glucose",
    "Heart_ stroke": "heart_disease"
}

missing_columns = [
    column
    for column in column_name_mapping
    if column not in df_clean.columns
]

if missing_columns:
    raise KeyError(
        f"Expected columns were not found: {missing_columns}"
    )

df_clean = df_clean.rename(columns=column_name_mapping)

print("Column names standardised successfully\n")

for index, column in enumerate(df_clean.columns, start=1):
    print(f"{index}. {column}")

Column names standardised successfully

1. gender
2. age
3. education
4. current_smoker
5. cigarettes_per_day
6. bp_meds
7. prevalent_stroke
8. prevalent_hypertension
9. diabetes
10. total_cholesterol
11. systolic_bp
12. diastolic_bp
13. bmi
14. heart_rate
15. glucose
16. heart_disease


In [17]:
clean_target = (
    df_clean["heart_disease"]
    .astype("string")
    .str.strip()
    .str.lower()
)

expected_target_values = {"no", "yes"}
actual_target_values = set(clean_target.dropna().unique())

unexpected_values = actual_target_values - expected_target_values

if unexpected_values:
    raise ValueError(
        f"Unexpected target values found: {unexpected_values}"
    )

df_clean["heart_disease"] = clean_target.map({
    "no": 0,
    "yes": 1
}).astype("int64")

target_check = (
    df_clean["heart_disease"]
    .value_counts()
    .sort_index()
    .rename_axis("Heart Disease")
    .reset_index(name="Count")
)

target_check

,Heart Disease,Count
0,0,3594
1,1,644


In [18]:
df_clean["gender"] = (
    df_clean["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
)

expected_gender_values = {"male", "female"}
actual_gender_values = set(df_clean["gender"].dropna().unique())

unexpected_gender_values = (
    actual_gender_values - expected_gender_values
)

if unexpected_gender_values:
    raise ValueError(
        f"Unexpected gender values found: "
        f"{unexpected_gender_values}"
    )

gender_summary = (
    df_clean["gender"]
    .value_counts(dropna=False)
    .rename_axis("Gender")
    .reset_index(name="Count")
)

gender_summary["Percentage"] = (
    gender_summary["Count"] / len(df_clean) * 100
).round(2)

gender_summary

,Gender,Count,Percentage
0,female,2419,57.08
1,male,1819,42.92


In [19]:
binary_columns = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

stroke_values = (
    df_clean["prevalent_stroke"]
    .astype("string")
    .str.strip()
    .str.lower()
)

stroke_mapping = {
    "no": 0,
    "yes": 1,
    "0": 0,
    "1": 1
}

unexpected_stroke_values = (
    set(stroke_values.dropna().unique())
    - set(stroke_mapping.keys())
)

if unexpected_stroke_values:
    raise ValueError(
        f"Unexpected prevalent_stroke values: "
        f"{unexpected_stroke_values}"
    )

df_clean["prevalent_stroke"] = (
    stroke_values
    .map(stroke_mapping)
    .astype("Int64")
)

binary_validation = []

for column in binary_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    ).astype("Int64")

    unique_values = set(
        df_clean[column].dropna().unique()
    )

    unexpected_values = unique_values - {0, 1}

    if unexpected_values:
        raise ValueError(
            f"Unexpected values found in {column}: "
            f"{unexpected_values}"
        )

    binary_validation.append({
        "Column": column,
        "Unique Values": sorted(unique_values),
        "Missing Values": int(
            df_clean[column].isna().sum()
        )
    })

binary_validation_summary = pd.DataFrame(binary_validation)

binary_validation_summary

,Column,Unique Values,Missing Values
0,current_smoker,"[0, 1]",0
1,bp_meds,"[0, 1]",53
2,prevalent_stroke,"[0, 1]",0
3,prevalent_hypertension,"[0, 1]",0
4,diabetes,"[0, 1]",0


In [20]:
education_values = (
    df_clean["education"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[\s_-]+", "", regex=True)
)

education_mapping = {
    "uneducated": "uneducated",
    "primaryschool": "primary_school",
    "graduate": "graduate",
    "postgraduate": "postgraduate"
}

unexpected_education_values = (
    set(education_values.dropna().unique())
    - set(education_mapping.keys())
)

if unexpected_education_values:
    raise ValueError(
        f"Unexpected education values found: "
        f"{unexpected_education_values}"
    )

df_clean["education"] = education_values.map(education_mapping)

continuous_columns = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

numeric_validation = []

for column in continuous_columns:
    missing_before = df_clean[column].isna().sum()

    converted_values = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    missing_after = converted_values.isna().sum()
    newly_created_missing = missing_after - missing_before

    if newly_created_missing > 0:
        raise ValueError(
            f"Numeric conversion created "
            f"{newly_created_missing} missing values "
            f"in '{column}'."
        )

    df_clean[column] = converted_values

    numeric_validation.append({
        "Column": column,
        "Data Type": str(df_clean[column].dtype),
        "Missing Values": int(df_clean[column].isna().sum())
    })

education_summary = (
    df_clean["education"]
    .value_counts(dropna=False)
    .rename_axis("Education")
    .reset_index(name="Count")
)

print("Education categories:")
display(education_summary)

print("Numerical column validation:")
display(pd.DataFrame(numeric_validation))

Education categories:


,Education,Count
0,uneducated,1720
1,primary_school,1253
2,graduate,687
3,postgraduate,473
4,NaN,105


Numerical column validation:


,Column,Data Type,Missing Values
0,age,int64,0
1,cigarettes_per_day,float64,29
2,total_cholesterol,float64,50
3,systolic_bp,float64,0
4,diastolic_bp,float64,0
5,bmi,float64,19
6,heart_rate,float64,1
7,glucose,float64,388


In [21]:
valid_ranges = {
    "age": (18, 100),
    "cigarettes_per_day": (0, 100),
    "total_cholesterol": (50, 700),
    "systolic_bp": (70, 300),
    "diastolic_bp": (40, 200),
    "bmi": (10, 80),
    "heart_rate": (30, 220),
    "glucose": (30, 600)
}

range_validation = []

for column, (minimum_allowed, maximum_allowed) in valid_ranges.items():
    non_missing_values = df_clean[column].dropna()

    invalid_mask = (
        (non_missing_values < minimum_allowed)
        | (non_missing_values > maximum_allowed)
    )

    invalid_count = int(invalid_mask.sum())

    range_validation.append({
        "Column": column,
        "Observed Minimum": non_missing_values.min(),
        "Observed Maximum": non_missing_values.max(),
        "Allowed Minimum": minimum_allowed,
        "Allowed Maximum": maximum_allowed,
        "Invalid Values": invalid_count
    })

range_validation_summary = pd.DataFrame(range_validation)

range_validation_summary

,Column,Observed Minimum,Observed Maximum,Allowed Minimum,Allowed Maximum,Invalid Values
0,age,32.00,70.0,18,100,0
1,cigarettes_per_day,0.00,70.0,0,100,0
2,total_cholesterol,107.00,696.0,50,700,0
3,systolic_bp,83.50,295.0,70,300,0
4,diastolic_bp,48.00,142.5,40,200,0
5,bmi,15.54,56.8,10,80,0
6,heart_rate,44.00,143.0,30,220,0
7,glucose,40.00,394.0,30,600,0
